<a href="https://colab.research.google.com/github/Rakshitha443/ITA-0610-MACHINE-LEARNING-/blob/main/UNIT_2_CO2_Assignment_II_SDG_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# SDG 6: TRANSFER LEARNING FOR WATER SOURCE SAFETY CLASSIFICATION
# Model Architecture: MobileNetV2 (Pre-trained on ImageNet)
# =====================================================================

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 1. SET RANDOM SEEDS FOR REPRODUCIBILITY
tf.random.set_seed(42)
np.random.seed(42)

# 2. DEFINE HYPERPARAMETERS & SIMULATE DATA SCENARIO
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_SAMPLES = 100  # Simulating a data-scarce rural dataset (100 total images)

# Create synthetic image tensors representing small rural dataset
X_synthetic = np.random.uniform(
    0, 255, size=(NUM_SAMPLES, 224, 224, 3)
).astype(np.float32)
# Binary labels: 1 = Safe Water Source, 0 = Unsafe/Contaminated Source
y_synthetic = np.random.randint(0, 2, size=(NUM_SAMPLES,))

# Split into 80% Training and 20% Validation
train_x, val_x = X_synthetic[:80], X_synthetic[80:]
train_y, val_y = y_synthetic[:80], y_synthetic[80:]

# Data preprocessing pipeline (MobileNetV2 scaling)
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_x = preprocess_input(train_x)
val_x = preprocess_input(val_x)

# 3. BUILD TRANSFER LEARNING MODEL
# Load MobileNetV2 pre-trained base without its top classification layer
base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)

# FREEZE THE BASE MODEL LAYERS (Preserve generic pre-trained features)
base_model.trainable = False

# Construct custom classification head for binary water safety prediction
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)  # Dropout for regularization against overfitting
outputs = layers.Dense(1, activation="sigmoid")(
    x
)  # Output: Probability of being Safe

model = models.Model(inputs, outputs)

# 4. COMPILE MODEL
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Recall(name="recall")],
)

# Display Architecture Summary
print("=" * 70)
print("          TRANSFER LEARNING MODEL ARCHITECTURE (SDG 6)          ")
print("=" * 70)
model.summary()

# 5. TRAIN MODEL ON SCARCE LOCAL DATA
print("\n" + "=" * 70)
print("     TRAINING FINE-TUNED CLASSIFICATION HEAD ON RURAL DATA      ")
print("=" * 70)

history = model.fit(
    train_x,
    train_y,
    epochs=10,
    batch_size=BATCH_SIZE,
    validation_data=(val_x, val_y),
    verbose=1,
)

# 6. EVALUATE PERFORMANCE
val_loss, val_acc, val_recall = model.evaluate(val_x, val_y, verbose=0)

print("\n" + "=" * 70)
print("                   FINAL EVALUATION RESULTS                    ")
print("=" * 70)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc * 100:.2f}%")
print(f"Validation Recall   : {val_recall * 100:.2f}% (Critical for safety)")
print("=" * 70)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
          TRANSFER LEARNING MODEL ARCHITECTURE (SDG 6)          


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


     TRAINING FINE-TUNED CLASSIFICATION HEAD ON RURAL DATA      
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.4875 - loss: 0.7322 - recall: 0.2647 - val_accuracy: 0.5000 - val_loss: 0.6840 - val_recall: 0.0000e+00
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 638ms/step - accuracy: 0.6750 - loss: 0.6665 - recall: 0.5000 - val_accuracy: 0.5000 - val_loss: 0.6734 - val_recall: 0.0000e+00
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 889ms/step - accuracy: 0.5250 - loss: 0.7268 - recall: 0.3824 - val_accuracy: 0.5000 - val_loss: 0.6783 - val_recall: 0.0000e+00
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 639ms/step - accuracy: 0.5500 - loss: 0.7270 - recall: 0.3235 - val_accuracy: 0.5000 - val_loss: 0.6925 - val_recall: 0.0000e+00
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 620ms/step - accuracy: 0.5875 - loss: 0.7172 - recall: 0.2647 - val_accuracy: 0.5000 - val_loss: 0.6810 - val_recall: 0.0000e+00
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 793ms/step - accuracy: 0.6250 - loss: 0.6601 - recall: 0.